In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [6]:
with open("transcript.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Characters loaded:", len(text))
print(text[:500])

Characters loaded: 29266
The initials GPT stand for Generative Pretrained Transformer.
So that first word is straightforward enough, these are bots that generate new text.
Pretrained refers to how the model went through a process of learning
from a massive amount of data, and the prefix insinuates that there's
more room to fine-tune it on specific tasks with additional training.
But the last word, that's the real key piece.
A transformer is a specific kind of neural network, a machine learning model,
and it's the core i


## Step 1b - Indexing (Text splitting)

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_text(text)
print(len(chunks))
print(chunks[0])

37
The initials GPT stand for Generative Pretrained Transformer.
So that first word is straightforward enough, these are bots that generate new text.
Pretrained refers to how the model went through a process of learning
from a massive amount of data, and the prefix insinuates that there's
more room to fine-tune it on specific tasks with additional training.
But the last word, that's the real key piece.
A transformer is a specific kind of neural network, a machine learning model,
and it's the core invention underlying the current boom in AI.
What I want to do with this video and the following chapters is go through a
visually-driven explanation for what actually happens inside a transformer.
We're going to follow the data that flows through it and go step by step.
There are many different kinds of models that you can build using transformers.
Some models take in audio and produce a transcript.
This sentence comes from a model going the other way around,


## Step 1c & 1d - (Embedding Generation and Storing in Vector space) 

In [8]:
embeddings = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_texts(chunks, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
vector_store.index_to_docstore_id

{0: '8611f8f7-aad4-46b7-b88a-fa5bac1d3e3b',
 1: '0db041a1-5167-4c3f-9464-eb655384ebb3',
 2: '93d779cd-4b39-49ed-81f3-8eef1ccb093f',
 3: '1e5e6e35-01ef-44ca-a4d2-ed531dc8ce47',
 4: 'bc542411-d66c-41e9-904f-58d703c8c5e8',
 5: '6db21d75-d39f-45ad-9030-91b7a547913b',
 6: '7e73a07c-3603-48cd-affd-72d2a2619884',
 7: 'f5e75ae2-d0b2-4438-ad00-2451b6f4fef5',
 8: '89e6c9be-f117-4e5f-bac5-74fb750baee5',
 9: '069a2cb2-04cb-4250-8ad0-bd228c7388fa',
 10: 'eafef0bd-2bb0-4d12-acc6-0ca747d2e123',
 11: 'c5980f7f-a5fa-4951-9dd7-70e74e49c20e',
 12: 'aed6f5c4-3f85-4c09-bcd7-a759fffd858d',
 13: '1e128205-dc87-4ffe-945a-ac213a6acd3b',
 14: 'f814186b-09bb-4a00-868f-4bc9f9d724ad',
 15: '79fc3977-55eb-45a0-8679-d9dfb948650b',
 16: 'a197dddf-59ba-4805-8a97-af6b6119ec8e',
 17: 'c4f98601-acdc-4801-8412-140fe90f7fb2',
 18: '15cd27e0-b897-4186-8f9b-ea4847fb97df',
 19: '783527cd-e4c9-4791-8459-c8742e1822b6',
 20: 'dfcfe4a3-bdf3-4800-b41d-4408df604d14',
 21: '1bdd38ae-9494-4980-a46b-df8292fd8565',
 22: '342e0b6c-9bbf-

In [10]:
vector_store.get_by_ids(['8fdec002-1aad-46c5-9137-40476e6b98c7'])

[]

## Step 2 - (Retreival)

In [11]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs = {"k" : 4})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000018E69E48D70>, search_kwargs={'k': 4})

In [12]:
retriever.invoke("What's full form of GPT?")

[Document(id='8611f8f7-aad4-46b7-b88a-fa5bac1d3e3b', metadata={}, page_content="The initials GPT stand for Generative Pretrained Transformer.\nSo that first word is straightforward enough, these are bots that generate new text.\nPretrained refers to how the model went through a process of learning\nfrom a massive amount of data, and the prefix insinuates that there's\nmore room to fine-tune it on specific tasks with additional training.\nBut the last word, that's the real key piece.\nA transformer is a specific kind of neural network, a machine learning model,\nand it's the core invention underlying the current boom in AI.\nWhat I want to do with this video and the following chapters is go through a\nvisually-driven explanation for what actually happens inside a transformer.\nWe're going to follow the data that flows through it and go step by step.\nThere are many different kinds of models that you can build using transformers.\nSome models take in audio and produce a transcript.\nThis

## Step 3 - (Augmentation)

In [13]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [18]:
Prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables=["context", "question"]
)

In [19]:
question = "is GPT a transformer?"
restreived_docs = retriever.invoke(question)

In [22]:
restreived_docs

[Document(id='8611f8f7-aad4-46b7-b88a-fa5bac1d3e3b', metadata={}, page_content="The initials GPT stand for Generative Pretrained Transformer.\nSo that first word is straightforward enough, these are bots that generate new text.\nPretrained refers to how the model went through a process of learning\nfrom a massive amount of data, and the prefix insinuates that there's\nmore room to fine-tune it on specific tasks with additional training.\nBut the last word, that's the real key piece.\nA transformer is a specific kind of neural network, a machine learning model,\nand it's the core invention underlying the current boom in AI.\nWhat I want to do with this video and the following chapters is go through a\nvisually-driven explanation for what actually happens inside a transformer.\nWe're going to follow the data that flows through it and go step by step.\nThere are many different kinds of models that you can build using transformers.\nSome models take in audio and produce a transcript.\nThis

In [20]:
context = "\n\n".join(doc.page_content for doc in restreived_docs)

In [21]:
final_prompt = Prompt.invoke({"context" : context, "question" : question})

## Step 4 - (Generation)

In [23]:
answer = llm.invoke(final_prompt)
print(answer)

content='Yes. GPT (Generative Pre‑trained Transformer) is a transformer‑based model. The transcript explains that GPT’s name literally includes “Transformer,” and it describes GPT as a specific variant of the transformer architecture that is trained to generate text.' additional_kwargs={'reasoning_content': 'We need to answer based on transcript context. The question: "is GPT a transformer?" The transcript says: "The initials GPT stand for Generative Pretrained Transformer. So that first word is straightforward enough, these are bots that generate new text. ... A transformer is a specific kind of neural network, a machine learning model, and it\'s the core invention underlying the current boom in AI." Also: "the variant that you and I will focus on, which is the type that underlies tools like ChatGPT, will be a model that\'s trained to take in a piece of text...". So GPT is a transformer. So answer: Yes, GPT is a transformer model. Provide explanation.'} response_metadata={'token_usage

In [24]:
print(answer.content)

Yes. GPT (Generative Pre‑trained Transformer) is a transformer‑based model. The transcript explains that GPT’s name literally includes “Transformer,” and it describes GPT as a specific variant of the transformer architecture that is trained to generate text.


# Building Chain

In [25]:
def format_code(restreived_docs):
    context = "\n\n".join(doc.page_content for doc in restreived_docs)
    return context

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

parallel_chain = ({
    "context" : retriever | RunnableLambda(format_code),
    "question" : RunnablePassthrough()
})



In [27]:
parser = StrOutputParser()

main_chain = parallel_chain | Prompt | llm | parser

In [28]:
main_chain.invoke("Can u summerize the transcript in 50 words")

'GPT‑2 uses tokenization to split text into chunks, maps them to embeddings, and applies a transformer to predict the next token via a softmax over a 50,000‑word vocabulary. By sampling from this distribution and appending results, the model generates new text, illustrating how large language models produce more coherent sentences.'